# Phase 1 Analysis Notebook (Draft)
Lightweight inspection notebook for `data/10_mention_detection` outputs.

This is intentionally limited before Phase 3 disambiguation and deduplication decisions are finalized.

## 1) Project Setup
Resolve repository paths and load phase contracts.

In [11]:
from pathlib import Path
import sys
import re

import pandas as pd


def find_repo_root(start: Path) -> Path:
    cur = start.resolve()
    for _ in range(8):
        if (cur / "data").exists() and (cur / "speakermining" / "src").exists():
            return cur
        if cur.parent == cur:
            break
        cur = cur.parent
    raise RuntimeError("Repository root not found.")


ROOT = find_repo_root(Path.cwd())
SRC = ROOT / "speakermining" / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from process.mention_detection.config import (
    EPISODE_COLUMNS,
    FILE_EPISODES,
    FILE_PERSON_MENTIONS,
    FILE_PUBLIKATION,
    FILE_SEASONS,
    FILE_TOPIC_MENTIONS,
    PERSON_MENTION_COLUMNS,
    PHASE_DIR,
    PUBLIKATION_COLUMNS,
    SEASON_COLUMNS,
    TOPIC_MENTION_COLUMNS,
)

phase_dir = ROOT / PHASE_DIR
phase_dir

WindowsPath('C:/workspace/git/borgnetzwerk/speaker-mining/data/10_mention_detection')

## 2) Load Phase 1 Tables
Missing files are represented as empty dataframes with contract columns.

In [12]:
def load_or_empty(path: Path, columns: list[str]) -> pd.DataFrame:
    if not path.exists():
        return pd.DataFrame(columns=columns)
    df = pd.read_csv(path)
    for column in columns:
        if column not in df.columns:
            df[column] = pd.NA
    return df

episodes_df = load_or_empty(phase_dir / FILE_EPISODES, EPISODE_COLUMNS)
publications_df = load_or_empty(phase_dir / FILE_PUBLIKATION, PUBLIKATION_COLUMNS)
seasons_df = load_or_empty(phase_dir / FILE_SEASONS, SEASON_COLUMNS)
persons_df = load_or_empty(phase_dir / FILE_PERSON_MENTIONS, PERSON_MENTION_COLUMNS)
topics_df = load_or_empty(phase_dir / FILE_TOPIC_MENTIONS, TOPIC_MENTION_COLUMNS)

{
    'episodes_rows': len(episodes_df),
    'publications_rows': len(publications_df),
    'seasons_rows': len(seasons_df),
    'persons_rows': len(persons_df),
    'topics_rows': len(topics_df),
}

{'episodes_rows': 2036,
 'publications_rows': 2542,
 'seasons_rows': 17,
 'persons_rows': 10381,
 'topics_rows': 10713}

## 3) Lightweight Coverage Overview

In [13]:
overview = pd.DataFrame([
    {
        'table': 'episodes',
        'rows': len(episodes_df),
        'unique_ids': episodes_df['episode_id'].nunique() if 'episode_id' in episodes_df.columns else 0,
    },
    {
        'table': 'publications',
        'rows': len(publications_df),
        'unique_ids': publications_df['publikation_id'].nunique() if 'publikation_id' in publications_df.columns else 0,
    },
    {
        'table': 'seasons',
        'rows': len(seasons_df),
        'unique_ids': seasons_df['season_id'].nunique() if 'season_id' in seasons_df.columns else 0,
    },
    {
        'table': 'persons',
        'rows': len(persons_df),
        'unique_ids': persons_df['mention_id'].nunique() if 'mention_id' in persons_df.columns else 0,
    },
    {
        'table': 'topics',
        'rows': len(topics_df),
        'unique_ids': topics_df['mention_id'].nunique() if 'mention_id' in topics_df.columns else 0,
    },
])

overview

,table,rows,unique_ids
0,episodes,2036,2036
1,publications,2542,2542
2,seasons,17,17
3,persons,10381,10381
4,topics,10713,10713


## 4) Mention Quality Snapshot
Confidence-aware checks without assuming final entity resolution quality.

In [14]:
persons_conf = pd.to_numeric(persons_df['confidence'], errors='coerce') if 'confidence' in persons_df.columns else pd.Series(dtype='float64')
topics_conf = pd.to_numeric(topics_df['confidence'], errors='coerce') if 'confidence' in topics_df.columns else pd.Series(dtype='float64')

quality = {
    'persons_avg_confidence': float(persons_conf.mean()) if not persons_conf.empty else None,
    'persons_invalid_confidence_rows': int(persons_conf.isna().sum()) if not persons_conf.empty else 0,
    'persons_missing_beschreibung': int(persons_df['beschreibung'].isna().sum()) if 'beschreibung' in persons_df.columns else 0,
    'topics_avg_confidence': float(topics_conf.mean()) if not topics_conf.empty else None,
    'topics_invalid_confidence_rows': int(topics_conf.isna().sum()) if not topics_conf.empty else 0,
}

person_rules = persons_df['parsing_rule'].value_counts().rename_axis('parsing_rule').to_frame('rows') if 'parsing_rule' in persons_df.columns else pd.DataFrame()
topic_rules = topics_df['parsing_rule'].value_counts().rename_axis('parsing_rule').to_frame('rows') if 'parsing_rule' in topics_df.columns else pd.DataFrame()

quality, person_rules, topic_rules

({'persons_avg_confidence': 0.9106088045467681,
  'persons_invalid_confidence_rows': 0,
  'persons_missing_beschreibung': 765,
  'topics_avg_confidence': 0.7758078969476336,
  'topics_invalid_confidence_rows': 0},
                                   rows
 parsing_rule                          
 single_parenthetical              9143
 name_without_local_parenthetical   762
 last_name_parenthetical            283
 single_parenthetical_mononym       139
 group_parenthetical                 51
 surname_primary_no_parenthetical     3,
                                  rows
 parsing_rule                         
 labelled_topic_list              4671
 labelled_topic_list_with_commas  1856
 inline_label_comma_split         1659
 semicolon_part_comma_split       1421
 cue_based_clause                 1106)

## 5) Early Legacy-Inspired Sanity Checks (Draft)
These checks are only weak signals and should not drive final conclusions before later phases.

In [15]:
episodes_without_person_mentions = int(
    episodes_df['episode_id'].nunique() - persons_df['episode_id'].nunique()
) if ('episode_id' in episodes_df.columns and 'episode_id' in persons_df.columns) else None

names_ending_with_von = int(
    persons_df['name'].fillna('').str.upper().str.endswith(' VON').sum()
) if 'name' in persons_df.columns else None

thevessen_variant_count = int(
    episodes_df['infos'].fillna('').str.contains(r'THEVE(?:ß|SS)EN', regex=True, case=False).sum()
) if 'infos' in episodes_df.columns else None

{
    'episodes_without_person_mentions': episodes_without_person_mentions,
    'names_ending_with_von': names_ending_with_von,
    'infos_thevessen_variant_count': thevessen_variant_count,
}

if episodes_without_person_mentions is not None:
    episodes_with_mentions = set(persons_df['episode_id'].dropna().unique()) if 'episode_id' in persons_df.columns else set()
    episodes_without_mentions_df = episodes_df[episodes_df['episode_id'].isin(episodes_with_mentions) == False] if 'episode_id' in episodes_df.columns else pd.DataFrame()
    print("Episodes without person mentions:")
    display(episodes_without_mentions_df)

    # Persist this list as actionable feedback for mention-detection improvements.
    phase_dir.mkdir(parents=True, exist_ok=True)
    episodes_without_mentions_path = phase_dir / "episodes_without_person_mentions.csv"
    episodes_without_mentions_df.to_csv(episodes_without_mentions_path, index=False)
    print(f"Saved: {episodes_without_mentions_path}")

Episodes without person mentions:


,episode_id,sendungstitel,publikation_id,publikationsdatum,dauer,archivnummer,prod_nr_beitrag,zeit_tc_start,zeit_tc_end,season,staffel,folge,folgennr,infos
29,ep_e0ab0bb17652,Markus Lanz daheim: Mein Advent in Südtirol,pb_a6a95e5675a3,14.12.2008,102'12,4012257101,00542/00792,NaN,NaN,NaN,NaN,NaN,NaN,"Markus Lanz präsentiert die Sendung ""Markus La..."
454,ep_ec3af6e34bfd,Markus Lanz im Interview mit Peter Kohl und Wa...,NaN,01.03.2013,50'46,4023581501,00529/03695,10:00:00,10:50:46,NaN,NaN,NaN,NaN,NaN
784,ep_c3730ee7effd,Markus Lanz Wieder vereint!,pb_ec45b159e22e,29.09.2015,72'48,4031576001,00529/06263,10:00:00,11:12:48,"Markus Lanz, Staffel 8",8.0,95.0,781.0,10:00:00 - 11:12:48 072'48 Aus Anlass des 25.J...
853,ep_ef4aa5034a38,Hans-Dietrich Genscher im Gespräch mit Markus ...,pb_f4c8f4766435,05.04.2016,58'37,4033593601,00529/06661,01:09:10,02:07:46,NaN,NaN,NaN,NaN,01:09:10 - 02:07:46 058'36 Im Spätsommer 2015 ...
927,ep_dd267acf37b9,Markus Lanz - Amerika ungeschminkt,pb_fd70dee9832d,27.10.2016,84'32,4037009201,00529/07051,10:00:00,11:24:42,NaN,NaN,NaN,NaN,10:00:00 - 11:24:42 084'42 Markus Lanz macht s...
1009,ep_93c6078841d6,Markus Lanz 31.05.2017,pb_187d31008e77,31.05.2017,74'52,4040475801,00529/04262,23:27:44,00:42:36,"Markus Lanz, Staffel 10",10.0,61.0,1005.0,23:27:44 - 00:42:36 074'52 O-Ton Interview Mar...
1029,ep_53060ade5635,Markus Lanz - Kuba!,pb_bb81b58ac391,15.08.2017,83'45,4041884601,00529/07339,10:00:00,11:23:45,NaN,NaN,NaN,NaN,10:00:00 - 11:23:45 083'45 Im Dezember 2014 be...
1077,ep_5ce8317b9b11,Mit Markus Lanz im Heiligen Land,pb_44a77ae60406,25.12.2017,58'37,4045399001,00571/00016,10:00:00,10:58:36,NaN,NaN,NaN,NaN,10:00:00 - 10:58:36 058'36 Am ersten Weihnacht...
1141,ep_b9f71cf5eea6,Markus Lanz - Russland!,pb_9cb4d7b0ebea,14.06.2018,88'29,4050138801,00529/08675,00:00:00,01:28:29,NaN,NaN,NaN,NaN,00:00:00 - 01:28:29 088'29 Russland ist 2018 n...
1226,ep_dae15dfa2bc5,Markus Lanz 19.02.2019,pb_694b48513ae1,19.02.2019,75'38,4056283001,00529/08330,22:46:14,00:01:52,"Markus Lanz, Staffel 12",12.0,19.0,1218.0,NaN


Saved: C:\workspace\git\borgnetzwerk\speaker-mining\data\10_mention_detection\episodes_without_person_mentions.csv
